In [1]:
import os
import torch
import torch.utils.data as data
from collections import defaultdict
import random

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: idx for idx, aa in enumerate(AMINO_ACIDS)}


class SequenceFileDataset(data.Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform
        self.samples = []
        self.classes = []
        self.class_to_idx = {}

        filenames = sorted(
            [f for f in os.listdir(root_dir) if os.path.isfile(os.path.join(root_dir, f))]
        )

        for idx, fname in enumerate(filenames):
            self.classes.append(fname)
            self.class_to_idx[fname] = idx
            filepath = os.path.join(root_dir, fname)

            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    seq = line.strip()
                    if seq:
                        self.samples.append((seq, idx))

        self.num_classes = len(self.classes)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sequence, label = self.samples[idx]
        if self.transform is not None:
            sequence = self.transform(sequence)
        if self.target_transform is not None:
            label = self.target_transform(label)
        return sequence, label


def transform_sequence(seq):
    if isinstance(seq, bytes):
        seq = seq.decode("utf-8")
    seq = seq.strip().upper()
    if len(seq) != 9:
        raise ValueError(f"Sequence length must be 9, got {len(seq)}: {seq}")
    one_hot = torch.zeros(len(seq), len(AMINO_ACIDS), dtype=torch.float32)
    for i, aa in enumerate(seq):
        if aa in AA_TO_INDEX:
            one_hot[i, AA_TO_INDEX[aa]] = 1.0
    return one_hot.view(-1)


def encode_label_one_hot(label, num_classes):
    one_hot = torch.zeros(num_classes, dtype=torch.float32)
    one_hot[label] = 1.0
    return one_hot


def stratified_train_test_split(dataset, test_size=0.1, shuffle=True, seed=42):
    """
    Stratified split that preserves class proportions in both train and test.
    Works correctly even when negative class vastly outnumbers positives.
    """
    labels = [label for _, label in dataset.samples]
    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[label].append(idx)

    train_indices, test_indices = [], []
    rng = random.Random(seed)

    for label, inds in label_to_indices.items():
        if shuffle:
            rng.shuffle(inds)
        if len(inds) == 1:
            train_indices.append(inds[0])
        else:
            n_test = max(1, min(int(round(len(inds) * test_size)), len(inds) - 1))
            test_indices.extend(inds[:n_test])
            train_indices.extend(inds[n_test:])

    return train_indices, test_indices


# ── Build dataset ───────────────────────────────────────────────────────────
root = "ex1_data"
dataset = SequenceFileDataset(
    root,
    transform=transform_sequence,
)
dataset.target_transform = lambda label: encode_label_one_hot(label, dataset.num_classes)

train_indices, test_indices = stratified_train_test_split(dataset, test_size=0.1)
train_dataset = data.Subset(dataset, train_indices)
test_dataset  = data.Subset(dataset, test_indices)

print("classes:     ", dataset.classes)
print("class_to_idx:", dataset.class_to_idx)
print(f"total samples: {len(dataset)}")
print(f"train: {len(train_dataset)}  |  test: {len(test_dataset)}")

from collections import Counter
train_labels = [dataset.samples[i][1] for i in train_indices]
test_labels  = [dataset.samples[i][1] for i in test_indices]
print("Train class counts:", dict(sorted(Counter(train_labels).items())))
print("Test  class counts:", dict(sorted(Counter(test_labels).items())))


classes:      ['A0101_pos.txt', 'A0201_pos.txt', 'A0203_pos.txt', 'A0207_pos.txt', 'A0301_pos.txt', 'A2402_pos.txt', 'negs.txt']
class_to_idx: {'A0101_pos.txt': 0, 'A0201_pos.txt': 1, 'A0203_pos.txt': 2, 'A0207_pos.txt': 3, 'A0301_pos.txt': 4, 'A2402_pos.txt': 5, 'negs.txt': 6}
total samples: 37383
train: 33645  |  test: 3738
Train class counts: {0: 1142, 1: 2353, 2: 1645, 3: 2983, 4: 1493, 5: 1986, 6: 22043}
Test  class counts: {0: 127, 1: 261, 2: 183, 3: 331, 4: 166, 5: 221, 6: 2449}


In [2]:
import torch.nn as nn


class SimpleClassifier(nn.Module):

    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.linear1 = nn.Linear(in_features=num_inputs, out_features=128)
        self.linear2 = nn.Linear(in_features=128, out_features=64)
        self.linear3 = nn.Linear(in_features=64, out_features=num_outputs)
        self.act_fn  = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        x = self.act_fn(x)
        x = self.linear3(x)
        return x


model     = SimpleClassifier(num_inputs=180, num_outputs=7)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [3]:
from collections import Counter
import torch
import torch.nn as nn
import torch.utils.data as data

# ── Class-frequency weights for CrossEntropyLoss ───────────────────────────
# Formula: w_c = total / (num_classes * count_c)
# This gives rare classes (positives) higher weight and the dominant
# negative class lower weight, so the loss treats all classes fairly.
# NOTE: CrossEntropyLoss with one-hot labels requires float targets.
raw_labels = [y for _, y in dataset.samples]
counts      = Counter(raw_labels)
num_classes = dataset.num_classes
total       = sum(counts.values())

class_weights = torch.tensor(
    [total / (num_classes * counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)
print("Class weights:", class_weights)
print("Class counts: ", [counts[i] for i in range(num_classes)])

# CrossEntropyLoss is correct for mutually-exclusive multi-class classification.
# It accepts one-hot float targets directly (PyTorch >= 1.10).
# BCEWithLogitsLoss was WRONG here — it treats each logit as an independent
# binary decision, which is inappropriate for single-label classification.
loss_module = nn.CrossEntropyLoss(weight=class_weights)

# ── WeightedRandomSampler for the training loader ──────────────────────────
# Without this, each batch is ~96% negatives and the model never learns
# to distinguish positive classes from each other.
train_labels_list = [dataset.samples[i][1] for i in train_indices]
sample_weights    = torch.tensor(
    [1.0 / counts[lbl] for lbl in train_labels_list],
    dtype=torch.float32
)
sampler = data.WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(train_labels_list),
    replacement = True,
)
print("\nSampler ready — each class will appear roughly equally per epoch.")


Class weights: tensor([4.2084, 2.0430, 2.9215, 1.6115, 3.2191, 2.4198, 0.2180])
Class counts:  [1269, 2614, 1828, 3314, 1659, 2207, 24492]

Sampler ready — each class will appear roughly equally per epoch.


In [4]:
# Verify: with WeightedRandomSampler a few batches should show balanced classes
debug_loader = data.DataLoader(train_dataset, batch_size=512, sampler=sampler)
batch_x, batch_y = next(iter(debug_loader))
batch_class_ids = batch_y.argmax(dim=1)
print("Class distribution in one sampled batch:",
      dict(sorted(Counter(batch_class_ids.tolist()).items())))


Class distribution in one sampled batch: {0: 74, 1: 63, 2: 72, 3: 85, 4: 69, 5: 66, 6: 83}


In [5]:
import torch
import torch.utils.data as data
import numpy as np
from tqdm.auto import tqdm
import plotly.graph_objects as go
from IPython.display import display


def train_model_epoch_eval(
    model,
    optimizer,
    train_dataset,
    test_dataset,
    loss_module,
    sampler,
    num_epochs=150,
    batch_size=256,
    device="cpu",
):
    """
    Trains 'model' for 'num_epochs' and evaluates after each epoch.

    Training loss : CrossEntropyLoss with class weights (7-class objective).
    pos_neg_loss  : Binary cross-entropy on pos-vs-neg — a MONITORING metric
                    only, never used for gradient updates.
    """
    # WeightedRandomSampler overrides shuffle — do NOT pass shuffle=True
    train_loader = data.DataLoader(
        train_dataset, batch_size=batch_size, sampler=sampler
    )
    # Batched test loader — avoids OOM when the test set is large
    test_loader = data.DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False
    )

    # Separate binary loss for pos_neg monitoring (not used in training)
    pos_neg_bce = torch.nn.BCEWithLogitsLoss()

    train_losses   = []
    test_losses    = []
    pos_neg_losses = []
    model.to(device)

    # Negative class is the last class (negs.txt)
    NEG_CLASS_IDX = dataset.num_classes - 1

    fig = go.FigureWidget()
    fig.add_scatter(name="Train loss",   mode="lines+markers", x=[], y=[])
    fig.add_scatter(name="Test loss",    mode="lines+markers", x=[], y=[])
    fig.add_scatter(name="pos_neg_loss", mode="lines+markers", x=[], y=[])
    fig.update_layout(
        title="Loss over epochs",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        legend_title="Loss type",
    )
    display(fig)

    for epoch in tqdm(range(num_epochs)):

        # ── Training phase ──────────────────────────────────────────────────
        model.train()
        epoch_train_loss = []

        for data_inputs, data_labels in train_loader:
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)   # one-hot float [B, 7]

            preds = model(data_inputs)              # logits [B, 7]

            # CrossEntropyLoss accepts one-hot float targets directly
            loss = loss_module(preds, data_labels)
            epoch_train_loss.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_losses.append(float(np.mean(epoch_train_loss)))

        # ── Evaluation phase ────────────────────────────────────────────────
        model.eval()
        all_test_loss     = []
        all_pos_neg_logit = []
        all_pos_neg_true  = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)        # one-hot float [B, 7]

                logits = model(X_batch)             # [B, 7]

                # 7-class test loss (same objective as training)
                batch_test_loss = loss_module(logits, Y_batch)
                all_test_loss.append(batch_test_loss.item())

                # ── pos_neg monitoring metric ────────────────────────────
                # Binary: 0 = negative class, 1 = any positive class
                # We use the *negative* logit (class NEG_CLASS_IDX) as the
                # binary score: high => model thinks it's negative.
                # true_binary: 1 if sample is positive, 0 if negative
                true_class  = Y_batch.argmax(dim=1)                # [B]
                true_binary = (true_class != NEG_CLASS_IDX).float()  # 1=pos, 0=neg

                # Use negative-class logit as proxy for "is negative" score
                # Negate it so a high value => model thinks it is positive
                pos_logit = -logits[:, NEG_CLASS_IDX]               # [B]

                all_pos_neg_logit.append(pos_logit)
                all_pos_neg_true.append(true_binary)

        test_losses.append(float(np.mean(all_test_loss)))

        # pos_neg_loss: binary cross-entropy, lower = better discrimination
        # This is a MONITORING signal only — no gradients flow from it
        all_pn_logit = torch.cat(all_pos_neg_logit)
        all_pn_true  = torch.cat(all_pos_neg_true)
        pn_loss = pos_neg_bce(all_pn_logit, all_pn_true).item()
        pos_neg_losses.append(pn_loss)

        # Update live plot
        epochs_x = list(range(1, len(train_losses) + 1))
        fig.data[0].x, fig.data[0].y = epochs_x, train_losses
        fig.data[1].x, fig.data[1].y = epochs_x, test_losses
        fig.data[2].x, fig.data[2].y = epochs_x, pos_neg_losses

    display(fig)
    print("Training complete.")
    return train_losses, test_losses, pos_neg_losses


train_losses, test_losses, pos_neg_losses = train_model_epoch_eval(
    model,
    optimizer,
    train_dataset,
    test_dataset,
    loss_module,
    sampler,
)

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Train loss',
              'type': 'scatter',
              'uid': 'd5d25d55-49c0-4515-b697-bd2142d19805',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'Test loss',
              'type': 'scatter',
              'uid': '872c6396-eb5a-414b-887b-715bef72a0dd',
              'x': [],
              'y': []},
             {'mode': 'lines+markers',
              'name': 'pos_neg_loss',
              'type': 'scatter',
              'uid': 'c479f119-053f-49ca-ad8d-766e444234af',
              'x': [],
              'y': []}],
    'layout': {'legend': {'title': {'text': 'Loss type'}},
               'template': '...',
               'title': {'text': 'Loss over epochs'},
               'xaxis': {'title': {'text': 'Epoch'}},
               'yaxis': {'title': {'text': 'Loss'}}}
})

  0%|          | 0/150 [00:00<?, ?it/s]

/home/tabibon/deep_learning/ex1/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


KeyboardInterrupt: 

In [ ]:
# Quick evaluation after training
with torch.no_grad():
    eval_loader = data.DataLoader(test_dataset, batch_size=512, shuffle=False)
    model.eval()
    all_preds, all_true = [], []
    for X_batch, Y_batch in eval_loader:
        logits = model(X_batch)
        all_preds.append(logits.argmax(dim=1))
        all_true.append(Y_batch.argmax(dim=1))
    all_preds = torch.cat(all_preds)
    all_true  = torch.cat(all_true)

NEG_CLASS_IDX = dataset.num_classes - 1
overall_acc = (all_preds == all_true).float().mean().item()
pos_correct = ((all_preds == all_true) & (all_true != NEG_CLASS_IDX)).sum().item()
pos_total   = (all_true != NEG_CLASS_IDX).sum().item()
neg_correct = ((all_preds == all_true) & (all_true == NEG_CLASS_IDX)).sum().item()
neg_total   = (all_true == NEG_CLASS_IDX).sum().item()

print(f"Overall accuracy : {overall_acc:.4f}")
print(f"Positive accuracy: {pos_correct}/{pos_total} = {pos_correct/max(pos_total,1):.4f}")
print(f"Negative accuracy: {neg_correct}/{neg_total} = {neg_correct/max(neg_total,1):.4f}")
print(f"Final pos_neg_loss: {pos_neg_losses[-1]:.4f}  (epoch 1: {pos_neg_losses[0]:.4f})")